# 07. washer_dryer 보강 이미지 수집 (Naver Shopping API → staging)

**목적**: `config.NAVER_SEARCH_QUERIES['washer_dryer']` 쿼리로 보강 이미지 수집

**흐름**: Naver API → `data/staging/washer_dryer/{query}/` → (검수) → `data/processed/washer_dryer/`

**수집 후 할 일**: `08_approve_staging.ipynb`에서 이미지를 검수하고 processed로 이동하세요.

> `data/processed/`와 `data/raw/`는 이 노트북에서 절대 수정하지 않습니다.

In [ ]:
import os, sys, re, requests
from io import BytesIO
from pathlib import Path
from PIL import Image as PILImage
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import config

# Naver API 자격증명 (.env 또는 환경변수에서 로드)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
NAVER_ID     = os.getenv('NAVER_CLIENT_ID', '')
NAVER_SECRET = os.getenv('NAVER_CLIENT_SECRET', '')
if not NAVER_ID or not NAVER_SECRET:
    raise EnvironmentError('NAVER_CLIENT_ID / NAVER_CLIENT_SECRET 환경변수가 없습니다. .env를 확인하세요.')
print('Naver API 자격증명 확인 OK')

CLASS_NAME       = 'washer_dryer'
QUERIES          = config.NAVER_SEARCH_QUERIES[CLASS_NAME]
IMAGES_PER_QUERY = 30      # 쿼리당 수집 수 (최대 100)
MIN_IMG_SIZE     = 150     # 픽셀 — 이 미만은 저장 안 함
STAGING_DIR      = os.path.join('data', 'staging', CLASS_NAME)
METADATA_CSV     = os.path.join(config.METADATA_DIR, 'washer_dryer_staging_metadata.csv')

print(f'검색어 {len(QUERIES)}개  |  쿼리당 {IMAGES_PER_QUERY}장  |  최대 {len(QUERIES)*IMAGES_PER_QUERY}장')
for i, q in enumerate(QUERIES):
    print(f'  {i+1:2d}. {q}')

In [ ]:
def fetch_items(query, display):
    r = requests.get(
        'https://openapi.naver.com/v1/search/shop.json',
        headers={'X-Naver-Client-Id': NAVER_ID, 'X-Naver-Client-Secret': NAVER_SECRET},
        params={'query': query, 'display': display, 'start': 1},
        timeout=config.DOWNLOAD_TIMEOUT,
    )
    r.raise_for_status()
    return r.json().get('items', [])

def download_and_validate(url, save_path, min_px):
    try:
        r = requests.get(url, timeout=config.DOWNLOAD_TIMEOUT)
        if r.status_code != 200 or 'image' not in r.headers.get('Content-Type',''):
            return False, 0, 0
        img = PILImage.open(BytesIO(r.content)).convert('RGB')
        w, h = img.size
        if w < min_px or h < min_px:
            return False, w, h
        img.save(save_path, 'JPEG', quality=95)
        return True, w, h
    except Exception:
        return False, 0, 0

records  = []
n_saved  = 0
n_skip   = 0

for query in QUERIES:
    slug  = re.sub(r'[^\w가-힣]', '_', query)
    qdir  = os.path.join(STAGING_DIR, slug)
    os.makedirs(qdir, exist_ok=True)

    try:
        items = fetch_items(query, IMAGES_PER_QUERY)
    except Exception as e:
        print(f'[오류] {query}: {e}')
        continue

    saved_q = 0
    for idx, item in enumerate(tqdm(items, desc=query, leave=False)):
        img_url = item.get('image', '')
        if not img_url:
            continue
        fname     = f'stg_{idx:04d}.jpg'
        save_path = os.path.join(qdir, fname)
        if os.path.exists(save_path):    # 이미 다운로드됨
            continue

        ok, w, h = download_and_validate(img_url, save_path, MIN_IMG_SIZE)
        if ok:
            title_clean = re.sub(r'<[^>]+>', '', item.get('title', ''))
            records.append({
                'query':       query,
                'title':       title_clean[:120],
                'link':        item.get('link', ''),
                'image_url':   img_url,
                'saved_path':  save_path,
                'width':       w,
                'height':      h,
                'status':      'staged',   # staged | approved | rejected
                'collected_at': datetime.now().isoformat(timespec='seconds'),
            })
            saved_q += 1
            n_saved += 1
        else:
            n_skip += 1

    print(f'  [{query}] {saved_q}장 저장  |  {len(items)-saved_q}장 스킵')

print(f'\n수집 완료 — 저장 {n_saved}장  |  스킵(소형/오류) {n_skip}장')

In [ ]:
# 기존 메타데이터에 누적 저장 (재실행 시 중복 URL 제거)
new_df = pd.DataFrame(records)
if os.path.exists(METADATA_CSV):
    old_df = pd.read_csv(METADATA_CSV)
    combined = pd.concat([old_df, new_df], ignore_index=True)
    combined.drop_duplicates(subset=['image_url'], keep='last', inplace=True)
else:
    combined = new_df

combined.to_csv(METADATA_CSV, index=False, encoding='utf-8-sig')
print(f'메타데이터 저장: {METADATA_CSV}')
print(f'총 레코드: {len(combined)}건')
print()
print('=== 쿼리별 수집 결과 ===')
for q, grp in combined[combined['status']=='staged'].groupby('query'):
    print(f'  {q}: {len(grp)}장')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

staged_files = []
for q_slug in sorted(os.listdir(STAGING_DIR)):
    qd = os.path.join(STAGING_DIR, q_slug)
    if not os.path.isdir(qd):
        continue
    for f in sorted(os.listdir(qd)):
        if f.lower().endswith(('.jpg','.jpeg','.png')):
            staged_files.append((q_slug, os.path.join(qd, f)))

print(f'스테이징 총 {len(staged_files)}장')

# 쿼리별 샘플 5장씩 미리보기
by_query = {}
for slug, fpath in staged_files:
    by_query.setdefault(slug, []).append(fpath)

SAMPLE = 5
n_queries = len(by_query)
fig, axes = plt.subplots(n_queries, SAMPLE, figsize=(SAMPLE*3, n_queries*3))
if n_queries == 1:
    axes = [axes]

for row_i, (slug, paths) in enumerate(sorted(by_query.items())):
    for col_i in range(SAMPLE):
        ax = axes[row_i][col_i] if isinstance(axes[row_i], np.ndarray) else axes[row_i][col_i]
        if col_i < len(paths):
            try:
                ax.imshow(PILImage.open(paths[col_i]).convert('RGB'))
                ax.set_title(f'{slug[:18]}\n{os.path.basename(paths[col_i])}',
                             fontsize=6)
            except:
                ax.text(0.5, 0.5, 'ERR', ha='center', va='center',
                        transform=ax.transAxes)
        ax.axis('off')

plt.suptitle(f'수집 스테이징 — 쿼리별 샘플 {SAMPLE}장', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('다음 단계: 08_approve_staging.ipynb 에서 검수 후 processed로 이동')